# TimelineQA Experiments Colab

Run cells one by one. Start with the smoke test before running a full model evaluation.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Clone or pull repo
import os
import subprocess

repo_dir = '/content/timelineqa_experiments'
repo_url = 'https://github.com/KYPKRISHNAREDDY/timelineqa_experiments.git'

if os.path.exists(repo_dir):
    subprocess.run(['git', '-C', repo_dir, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, repo_dir], check=True)

os.chdir(repo_dir)
print('Current folder:', os.getcwd())

In [ ]:
# Cell 4: Install requirements
!pip install -r requirements.txt

In [ ]:
# Cell 5: Read HF_TOKEN from Colab Secrets and set cache paths
from google.colab import userdata
import os
import subprocess

hf_token = userdata.get('HF_TOKEN')
os.environ['HF_HOME'] = '/content/drive/MyDrive/timelineqa_hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/content/drive/MyDrive/timelineqa_hf_cache'

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    subprocess.run(['hf', 'auth', 'login', '--token', hf_token], check=True)
else:
    print('HF_TOKEN not found. Public models may still work, but gated models will fail.')

In [ ]:
# Cell 6: Clone official TimelineQA
!bash scripts/00_clone_timelineqa.sh

In [ ]:
# Cell 7: Make n=50 atomic sample
!python scripts/02_make_samples.py --task atomic --n 50 --seed 42

In [ ]:
# Cell 8: Smoke test Qwen 0.5B with --limit 3
!python scripts/04_run_model.py \
  --task atomic \
  --sample data/samples/atomic_n50.jsonl \
  --model_id Qwen/Qwen2.5-0.5B-Instruct \
  --backend hf \
  --retriever bm25 \
  --top_k 5 \
  --max_new_tokens 32 \
  --temperature 0 \
  --output outputs/predictions/atomic_n50_qwen05b_bm25.jsonl \
  --limit 3

In [ ]:
# Cell 9: Evaluate
!python scripts/05_evaluate.py \
  --predictions outputs/predictions/atomic_n50_qwen05b_bm25.jsonl \
  --task atomic \
  --output outputs/metrics/atomic_n50_qwen05b_bm25_metrics.json

In [ ]:
# Cell 10: Copy outputs to Drive
!mkdir -p /content/drive/MyDrive/timelineqa_results
!cp -r outputs /content/drive/MyDrive/timelineqa_results/

In [ ]:
# Cell 11: Show metrics
!cat outputs/metrics/atomic_n50_qwen05b_bm25_metrics.json